# MeaningFlow Notebook 01  
## Semantic Space → Coverage → Opportunity

#This notebook demonstrates the core MeaningFlow workflow:
1. Load semantic objects, demand signals, and structural edges
2. Construct a semantic graph
3. Measure demand-weighted semantic coverage
4. Identify high-value opportunity regions in meaning space

The goal is not prediction, but decision support.


In [ ]:
# CELL 2 — Imports (Code)

import pandas as pd
import numpy as np
import networkx as nx

from pathlib import Path


In [ ]:
#CELL 3 — Load Example Data (Code)

DATA_DIR = Path("../data/examples")

objects = pd.read_csv(DATA_DIR / "objects.csv")
demand = pd.read_csv(DATA_DIR / "demand.csv")
edges = pd.read_csv(DATA_DIR / "edges.csv")

objects.head(), demand.head(), edges.head()


(        id      type                                               text  \
 0  doc_001  DOCUMENT         Guide: Best trail running shoes for winter   
 1  doc_002  DOCUMENT              Product Category: Trail Running Shoes   
 2  doc_003  DOCUMENT  How to choose running shoes: stability vs neutral   
 3  qry_001     QUERY                    best trail running shoes winter   
 4  qry_002     QUERY                        trail running shoes near me   
 
                                        metadata_json  
 0  {"site_section":"blog","url":"/blog/winter-tra...  
 1  {"site_section":"category","url":"/trail-runni...  
 2  {"site_section":"blog","url":"/blog/stability-...  
 3                  {"source":"search","market":"US"}  
 4                  {"source":"search","market":"US"}  ,
   object_id    signal_type   value  start_date    end_date  \
 0   qry_001  SEARCH_VOLUME  5400.0  2025-10-01  2025-10-31   
 1   qry_002  SEARCH_VOLUME  2200.0  2025-10-01  2025-10-31   
 2   qry_003  SE

In [ ]:
# CELL 4 — Basic Object Separation (Code)
documents = objects[objects["type"] == "DOCUMENT"]
queries = objects[objects["type"] == "QUERY"]
entities = objects[objects["type"] == "ENTITY"]

len(documents), len(queries), len(entities)


(3, 3, 3)

In [ ]:
#CELL 5 — Build Semantic Graph (Code)

G = nx.DiGraph()

# Add nodes
for _, row in objects.iterrows():
    G.add_node(row["id"], type=row["type"])

# Add edges
for _, row in edges.iterrows():
    G.add_edge(
        row["src_id"],
        row["dst_id"],
        type=row["type"],
        weight=row["weight"]
    )

G.number_of_nodes(), G.number_of_edges()


(11, 11)

In [ ]:
#CELL 6 — Demand Aggregation (Code)
query_demand = (
    demand[demand["signal_type"] == "SEARCH_VOLUME"]
    .groupby("object_id")["value"]
    .sum()
)

query_demand


object_id
qry_001    5400.0
qry_002    2200.0
qry_003    1600.0
Name: value, dtype: float64

In [ ]:
#CELL: Config (Code)
USE_SENTENCE_TRANSFORMERS = True
MODEL_NAME = "all-MiniLM-L6-v2"
TOPK = 3
SIM_THRESHOLD = 0.55


### Step 3.1.0 — Prepare Text for Embeddings

Create `text_for_embedding` as the canonical text field used for embeddings.
If `text` is missing, fall back to `id` so every object can be embedded.


In [ ]:
import json

def parse_json(x):
    try:
        return json.loads(x) if isinstance(x, str) and x.strip() else {}
    except Exception:
        return {}

# If your CSV has metadata_json, parse it (optional but useful later)
if "metadata_json" in objects.columns:
    objects["metadata"] = objects["metadata_json"].apply(parse_json)
else:
    objects["metadata"] = [{} for _ in range(len(objects))]

# Ensure a text column exists
if "text" not in objects.columns:
    objects["text"] = ""

# Build text_for_embedding
objects["text_for_embedding"] = objects["text"].fillna("")
objects.loc[objects["text_for_embedding"].str.strip() == "", "text_for_embedding"] = objects["id"].astype(str)

objects[["id", "type", "text", "text_for_embedding"]].head()


,id,type,text,text_for_embedding
0,doc_001,DOCUMENT,Guide: Best trail running shoes for winter,Guide: Best trail running shoes for winter
1,doc_002,DOCUMENT,Product Category: Trail Running Shoes,Product Category: Trail Running Shoes
2,doc_003,DOCUMENT,How to choose running shoes: stability vs neutral,How to choose running shoes: stability vs neutral
3,qry_001,QUERY,best trail running shoes winter,best trail running shoes winter
4,qry_002,QUERY,trail running shoes near me,trail running shoes near me


In [ ]:
%pip install -q sentence-transformers scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install -q ipywidgets


In [ ]:
#CELL: Build Embeddings (Code)
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

texts = objects["text_for_embedding"].tolist()

if USE_SENTENCE_TRANSFORMERS:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME)
    X = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
else:
    rng = np.random.default_rng(42)
    dim = 384
    X = rng.normal(size=(len(objects), dim))
    X = X / np.linalg.norm(X, axis=1, keepdims=True)

id_to_idx = {obj_id: i for i, obj_id in enumerate(objects["id"].tolist())}
doc_ids = objects.loc[objects["type"] == "DOCUMENT", "id"].tolist()
qry_ids = objects.loc[objects["type"] == "QUERY", "id"].tolist()

doc_idx = np.array([id_to_idx[i] for i in doc_ids])
qry_idx = np.array([id_to_idx[i] for i in qry_ids])

S = cosine_similarity(X[qry_idx], X[doc_idx])
S.shape



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\bcurr\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bcurr\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(3, 3)

In [ ]:
CELL: Add Similarity Edges (Code)
for qi, qid in enumerate(qry_ids):
    sims = S[qi]
    topk_idx = np.argsort(-sims)[:TOPK]
    for j in topk_idx:
        score = float(sims[j])
        if score >= SIM_THRESHOLD:
            did = doc_ids[j]
            G.add_edge(qid, did, type="SIMILARITY", weight=score)

G.number_of_edges()


In [ ]:
CELL: Coverage + Opportunity (Code)
query_demand = (
    demand[demand["signal_type"] == "SEARCH_VOLUME"]
    .groupby("object_id")["value"]
    .sum()
)
query_demand = query_demand[query_demand.index.isin(qry_ids)]

covered_ids = []
uncovered_rows = []

for qi, qid in enumerate(qry_ids):
    if qid not in query_demand.index:
        continue
    best_sim = float(S[qi].max())
    if best_sim >= SIM_THRESHOLD:
        covered_ids.append(qid)
    else:
        d = float(query_demand.loc[qid])
        gap = max(0.0, SIM_THRESHOLD - best_sim)
        uncovered_rows.append({
            "query_id": qid,
            "best_sim": best_sim,
            "demand": d,
            "gap": gap,
            "opportunity": d * gap
        })

coverage_ratio = (query_demand.loc[covered_ids].sum() / query_demand.sum()) if len(query_demand) else np.nan
coverage_ratio


In [ ]:
#CELL: Display Ranked Opportunities (Code)
opportunity_df = pd.DataFrame(uncovered_rows).sort_values("opportunity", ascending=False)

opportunity_df = opportunity_df.merge(
    objects[objects["type"] == "QUERY"][["id", "text_for_embedding"]],
    left_on="query_id", right_on="id", how="left"
)

opportunity_df[["query_id","text_for_embedding","demand","best_sim","gap","opportunity"]]


In [ ]:
#CELL 7 — Coverage Proxy (Simple v0)
def covered_queries(graph, doc_ids):
    covered = set()
    for q in query_demand.index:
        for d in doc_ids:
            if graph.has_edge(q, d) or graph.has_edge(d, q):
                covered.add(q)
    return covered

doc_ids = set(documents["id"])
covered = covered_queries(G, doc_ids)

coverage_ratio = (
    query_demand.loc[list(covered)].sum() / query_demand.sum()
)

coverage_ratio


In [ ]:
#CELL 8 — Opportunity Scoring (Code)
uncovered = set(query_demand.index) - covered

opportunity = (
    query_demand.loc[list(uncovered)]
    .sort_values(ascending=False)
)

opportunity


In [ ]:
### Interpretation

- Covered queries represent semantic regions already supported by content.
- Uncovered queries represent demand-weighted opportunity.
- This simple proxy demonstrates how structure + demand identifies investment priorities.

In future iterations:
- Replace edge-based coverage with embedding distance
- Introduce authority weighting
- Simulate counterfactual content additions


In [ ]:
MeaningFlow reframes SEO and content optimization as a
**semantic capital allocation problem**.

Rather than optimizing pages, we identify where meaning,
structure, and demand are misaligned—and quantify the
economic opportunity of correcting that misalignment.

### Dependencies
This notebook optionally uses `sentence-transformers` for embeddings.
If unavailable, set `USE_SENTENCE_TRANSFORMERS = False` to run with deterministic random embeddings.
